# 1일차 실습 2. 기본 서버 처리 로직 구현

## 실습 목표

이번 실습에서는 실제 FastAPI 서버를 만들기 전에, 서버 내부에서 일어나는 처리 흐름을 Python 함수로 구현합니다.

핵심 흐름은 다음과 같습니다.

```text
입력 받기 → 입력값 확인하기 → 처리하기 → 응답 만들기
```

이번 노트북은 두 부분으로 구성됩니다.

1. 먼저 예제 코드를 실행하며 서버 처리 흐름을 관찰합니다.
2. 마지막 TODO에서 작은 handler 함수들을 직접 구현합니다.


## 1. 요청 데이터를 Python dict로 표현하기

API 요청 Body로 들어오는 JSON은 Python 코드에서는 dict처럼 다룰 수 있습니다.

아래 `request_data`는 사용자가 서버에 보낸 요청 Body라고 생각하면 됩니다.


In [7]:
from pprint import pprint

request_data = {
    "message": "hello api"
}

print(request_data)
print(request_data["message"])

{'message': 'hello api'}
hello api


## 2. 가장 단순한 handler 함수

handler는 요청 데이터를 받아 처리하고 응답을 만드는 함수입니다.

아래 함수는 요청 데이터에서 `message`를 꺼내 대문자로 변환한 뒤 dict 형태로 반환합니다.


In [8]:
def handle_request(data):
    message = data["message"]
    result = message.upper()

    return {
        "result": result
    }


response = handle_request(request_data)
print(response)

{'result': 'HELLO API'}


같은 handler에 여러 입력을 넣어보면서, 입력에 따라 응답이 어떻게 달라지는지 확인합니다.


In [9]:
examples = [
    {"message": "backend api"},
    {"message": "python handler"},
    {"message": "json response"}
]

for example in examples:
    print("입력:", example)
    print("응답:", handle_request(example))
    print("-" * 40)

입력: {'message': 'backend api'}
응답: {'result': 'BACKEND API'}
----------------------------------------
입력: {'message': 'python handler'}
응답: {'result': 'PYTHON HANDLER'}
----------------------------------------
입력: {'message': 'json response'}
응답: {'result': 'JSON RESPONSE'}
----------------------------------------


## 3. 입력값이 없을 때 생기는 문제

요청 데이터에 `message`가 없으면 handler가 정상적으로 처리할 수 없습니다.

실제 서버 로직에서는 들어온 입력값을 먼저 확인해야 합니다.


In [10]:
bad_request_data = {}

# 아래 코드는 KeyError를 발생시킬 수 있으므로 주석 처리합니다.
# handle_request(bad_request_data)

## 4. 입력 검증 추가하기

`data.get("message")`를 사용하면 key가 없을 때 오류 대신 `None`을 받을 수 있습니다.

이제 `message`가 없거나 비어 있으면 에러 응답을 반환하도록 바꿔봅니다.


In [11]:
def handle_request_with_validation(data):
    message = data.get("message")

    if not message:
        return {
            "error": "message 값이 필요합니다."
        }

    result = message.upper()

    return {
        "result": result
    }


print(handle_request_with_validation({"message": "hello api"}))
print(handle_request_with_validation({}))
print(handle_request_with_validation({"message": ""}))

{'result': 'HELLO API'}
{'error': 'message 값이 필요합니다.'}
{'error': 'message 값이 필요합니다.'}


## 5. 응답 형식 통일하기

API 응답 구조가 매번 다르면 클라이언트가 처리하기 어렵습니다.

성공 응답과 실패 응답을 비슷한 구조로 맞춰봅니다.


In [12]:
def handle_request_v2(data):
    message = data.get("message")

    if not message:
        return {
            "success": False,
            "error": "message 값이 필요합니다.",
            "data": None
        }

    result = message.upper()

    return {
        "success": True,
        "error": None,
        "data": {
            "result": result
        }
    }


print(handle_request_v2({"message": "hello api"}))
print(handle_request_v2({}))

{'success': True, 'error': None, 'data': {'result': 'HELLO API'}}
{'success': False, 'error': 'message 값이 필요합니다.', 'data': None}


## 6. Status Code까지 함께 표현하기

실제 HTTP 응답에는 Status Code가 포함됩니다.

아직 실제 서버를 실행하지는 않지만, dict 안에 `status_code`를 넣어 HTTP 응답 구조를 흉내낼 수 있습니다.


In [13]:
def handle_request_v3(data):
    message = data.get("message")

    if not message:
        return {
            "status_code": 400,
            "body": {
                "success": False,
                "error": "message 값이 필요합니다.",
                "data": None
            }
        }

    result = message.upper()

    return {
        "status_code": 200,
        "body": {
            "success": True,
            "error": None,
            "data": {
                "result": result
            }
        }
    }


print(handle_request_v3({"message": "hello api"}))
print(handle_request_v3({}))

{'status_code': 200, 'body': {'success': True, 'error': None, 'data': {'result': 'HELLO API'}}}
{'status_code': 400, 'body': {'success': False, 'error': 'message 값이 필요합니다.', 'data': None}}


## 7. 가짜 LLM 응답 함수 만들기

아직 실제 LLM API를 호출하지 않습니다.

대신 사용자 질문을 받아 가짜 답변을 반환하는 함수를 만들어봅니다.


In [14]:
def fake_llm_response(message):
    return f"'{message}'에 대한 가짜 LLM 응답입니다."


print(fake_llm_response("FastAPI가 뭐야?"))

'FastAPI가 뭐야?'에 대한 가짜 LLM 응답입니다.


## 8. 챗 API 형태의 handler 만들기

사용자 질문을 입력받고, 가짜 LLM 응답을 만들어 반환하는 handler를 만들어봅니다.

이 함수는 나중에 만들 `/chat` API의 내부 처리 흐름을 단순하게 흉내낸 것입니다.


In [15]:
def chat_handler(data):
    message = data.get("message")

    if not message:
        return {
            "status_code": 400,
            "body": {
                "success": False,
                "error": "message 값이 필요합니다.",
                "data": None
            }
        }

    answer = fake_llm_response(message)

    return {
        "status_code": 200,
        "body": {
            "success": True,
            "error": None,
            "data": {
                "answer": answer
            }
        }
    }


print(chat_handler({"message": "FastAPI가 뭐야?"}))
print(chat_handler({}))

{'status_code': 200, 'body': {'success': True, 'error': None, 'data': {'answer': "'FastAPI가 뭐야?'에 대한 가짜 LLM 응답입니다."}}}
{'status_code': 400, 'body': {'success': False, 'error': 'message 값이 필요합니다.', 'data': None}}


## 9. 요청 전체 구조 흉내 내기

실제 HTTP 요청에는 Method, Path, Body가 포함됩니다.

아래처럼 요청 전체를 dict로 표현해볼 수 있습니다.


In [16]:
api_request = {
    "method": "POST",
    "path": "/chat",
    "body": {
        "message": "API 서버가 하는 일을 알려줘"
    }
}

print(api_request)

{'method': 'POST', 'path': '/chat', 'body': {'message': 'API 서버가 하는 일을 알려줘'}}


## 10. 간단한 API 서버 흐름 만들기

method와 path를 확인한 뒤 알맞은 handler로 요청을 넘기는 구조를 만들어봅니다.


In [17]:
def simple_api_server(request):
    method = request.get("method")
    path = request.get("path")
    body = request.get("body", {})

    if method == "POST" and path == "/chat":
        return chat_handler(body)

    return {
        "status_code": 404,
        "body": {
            "success": False,
            "error": "요청한 API를 찾을 수 없습니다.",
            "data": None
        }
    }


print(simple_api_server(api_request))
print(simple_api_server({"method": "GET", "path": "/unknown", "body": {}}))

{'status_code': 200, 'body': {'success': True, 'error': None, 'data': {'answer': "'API 서버가 하는 일을 알려줘'에 대한 가짜 LLM 응답입니다."}}}
{'status_code': 404, 'body': {'success': False, 'error': '요청한 API를 찾을 수 없습니다.', 'data': None}}


# TODO 실습

지금까지는 서버 내부 처리 흐름을 실행하면서 확인했습니다.

이제 작은 함수들을 직접 구현해봅니다.

TODO는 앞에서 본 예제를 이해했는지 확인하는 테스트에 가깝습니다.  
예제 코드를 복사하기보다는, 각 함수가 맡은 역할을 생각하면서 직접 완성해보세요.


## TODO 1. 입력 검증 함수 만들기

### 목표

요청 데이터에서 `message` 값을 검증하는 `validate_message(data)` 함수를 완성하세요.

### 반환 규칙

- 검증 성공: `(True, None)`
- 검증 실패: `(False, 실패 이유)`

### 세부 구현 TODO

- TODO 1-1: `message` 값이 아예 없는 경우를 처리합니다.
- TODO 1-2: `message`가 문자열이 아닌 경우를 처리합니다.
- TODO 1-3: `message`가 빈 문자열이거나 공백뿐인 문자열인 경우를 처리합니다.
- TODO 1-4: 위 조건을 모두 통과하면 검증 성공 결과를 반환합니다.

### 힌트

- `data.get("message")`
- `isinstance(message, str)`
- `message.strip()`


In [18]:
def validate_message(data):
    message = data.get("message")

    # TODO 1-1:
    # message 값이 아예 없는 경우를 처리하세요.
    if message is None:
        return False, "message 값이 필요합니다."

    # TODO 1-2:
    # message가 문자열이 아닌 경우를 처리하세요.
    if not isinstance(message, str):
        return False, "message 값은 문자열이어야 합니다."   
    
    # TODO 1-3:
    # message가 빈 문자열이거나 공백뿐인 문자열인 경우를 처리하세요.
    if message.strip() == "":
        return False, "message 값은 빈 문자열이거나 공백만으로 구성될 수 없습니다."

    # TODO 1-4:
    # 위 조건을 모두 통과하면 검증 성공 결과를 반환하세요.
    return True, "검증 성공"


# 테스트
test_cases = [
    {"message": "hello"},
    {"message": ""},
    {"message": "   "},
    {},
    {"message": 123},
]

for i, case in enumerate(test_cases, start=1):
    print("=" * 60)
    print(f"테스트 케이스 {i}")
    print("입력:", case)

    is_valid, error_message = validate_message(case)

    if is_valid:
        print("검증 결과: 성공")
    else:
        print("검증 결과: 실패")
        print("실패 이유:", error_message)


테스트 케이스 1
입력: {'message': 'hello'}
검증 결과: 성공
테스트 케이스 2
입력: {'message': ''}
검증 결과: 실패
실패 이유: message 값은 빈 문자열이거나 공백만으로 구성될 수 없습니다.
테스트 케이스 3
입력: {'message': '   '}
검증 결과: 실패
실패 이유: message 값은 빈 문자열이거나 공백만으로 구성될 수 없습니다.
테스트 케이스 4
입력: {}
검증 결과: 실패
실패 이유: message 값이 필요합니다.
테스트 케이스 5
입력: {'message': 123}
검증 결과: 실패
실패 이유: message 값은 문자열이어야 합니다.


## TODO 2. 공통 응답 생성 함수 만들기

### 목표

성공 응답과 실패 응답을 일관된 형태로 만들기 위한 `make_response()` 함수를 완성하세요.

### 함수 입력

```python
make_response(success, data=None, error=None, status_code=200)
```

### 요구사항

- 반환값은 HTTP 응답을 흉내 낸 dict입니다.
- 상태 코드를 확인할 수 있어야 합니다.
- 실제 응답 내용은 별도의 body 영역에 담기도록 구성합니다.
- body 안에는 성공 여부, 실제 데이터, 에러 메시지를 구분해서 넣습니다.

### 힌트

앞에서 실습한 `handle_request_v3()`의 응답 모양을 참고하세요.


In [19]:
def make_response(success, data=None, error=None, status_code=200):
    # TODO:
    # success, data, error, status_code를 사용해 응답 dict를 구성하세요.
    response = {
        "success": success,
        "data": data,
        "error": error,
        "status_code": status_code
    }
    return response


# 테스트
success_response = make_response(True, data={"result": "HELLO"})
error_response = make_response(False, error="message 값이 필요합니다.", status_code=400)

print("=" * 60)
print("성공 응답")
pprint(success_response)

print("=" * 60)
print("실패 응답")
pprint(error_response)


성공 응답
{'data': {'result': 'HELLO'},
 'error': None,
 'status_code': 200,
 'success': True}
실패 응답
{'data': None,
 'error': 'message 값이 필요합니다.',
 'status_code': 400,
 'success': False}


## TODO 3. 챗 handler 다시 만들기

### 목표

앞에서 만든 `validate_message()`와 `make_response()`를 사용해 `chat_handler_v2(data)`를 완성하세요.

### 요구사항

- 반드시 `validate_message()`로 입력값을 먼저 검증합니다.
- 검증 실패 시 `make_response()`를 사용해 실패 응답을 반환합니다.
- 정상 입력이면 가짜 LLM 응답 함수를 호출합니다.
- 최종 응답도 `make_response()`를 사용해 반환합니다.
- 응답 데이터에는 `answer` 필드를 포함합니다.

### 세부 구현 TODO

- TODO 3-1: 입력값을 검증합니다.
- TODO 3-2: 검증 실패 응답을 반환합니다.
- TODO 3-3: 정상 입력일 때 가짜 LLM 응답을 생성합니다.
- TODO 3-4: `answer`를 포함한 성공 응답을 반환합니다.

> 이 TODO는 TODO 1의 `validate_message()`와 TODO 2의 `make_response()`를 사용합니다.


In [20]:
# 사용자의 입력을 받아 가짜 LLM 응답을 반환하는 함수
def fake_llm_response_for_todo(message):
    return f"'{message}'에 대한 가짜 LLM 응답입니다."


def chat_handler_v2(data):
    # TODO 3-1:
    # validate_message(data)를 사용해 입력값을 검증하세요.
    is_valid, error_message = validate_message(data)

    # TODO 3-2:
    # 검증 실패 시 make_response()를 사용해 실패 응답을 반환하세요.
    if not is_valid:
        return make_response(False, data=None, error=error_message, status_code=400)

    # TODO 3-3:
    # 정상 입력이면 fake_llm_response_for_todo(message)를 호출하세요.
    message = data["message"]
    answer = fake_llm_response_for_todo(message)

    # TODO 3-4:
    # answer를 data에 담아 make_response()로 성공 응답을 반환하세요.
    return make_response(True, data={"result": answer}, error=None, status_code=200)


# 테스트
test_cases = [
    {"message": "FastAPI가 뭐야?"},
    {"message": ""},
    {}
]

for i, case in enumerate(test_cases, start=1):
    print("=" * 60)
    print(f"테스트 케이스 {i}")
    print("입력:", case)
    print("응답:")
    pprint(chat_handler_v2(case))


테스트 케이스 1
입력: {'message': 'FastAPI가 뭐야?'}
응답:
{'data': {'result': "'FastAPI가 뭐야?'에 대한 가짜 LLM 응답입니다."},
 'error': None,
 'status_code': 200,
 'success': True}
테스트 케이스 2
입력: {'message': ''}
응답:
{'data': None,
 'error': 'message 값은 빈 문자열이거나 공백만으로 구성될 수 없습니다.',
 'status_code': 400,
 'success': False}
테스트 케이스 3
입력: {}
응답:
{'data': None,
 'error': 'message 값이 필요합니다.',
 'status_code': 400,
 'success': False}


## TODO 4. `/chat` 요청을 처리하는 서버 함수 만들기

### 목표

method와 path를 보고 `/chat` 요청을 처리하는 `simple_api_server_v2(request)`를 완성하세요.

### 요구사항

- 요청 dict에서 `method`, `path`, `body`를 꺼냅니다.
- `POST /chat` 요청이면 `chat_handler_v2(body)`를 호출합니다.
- 그 외 요청은 404 응답을 반환합니다.


In [22]:
def simple_api_server_v2(request):
    method = request.get("method")
    path = request.get("path")
    body = request.get("body", {})

    # TODO:
    # POST /chat 요청이면 chat_handler_v2(body)를 반환하세요.
    # 그 외 요청이면 404 응답을 반환하세요.
    if method == "POST" and path == "/chat":
        return chat_handler_v2(body)
    return make_response(False, data=None, error="요청 경로를 찾을 수 없습니다.", status_code=404)


# 테스트
requests_to_test = [
    {
        "method": "POST",
        "path": "/chat",
        "body": {"message": "FastAPI가 뭐야?"}
    },
    {
        "method": "POST",
        "path": "/chat",
        "body": {}
    },
    {
        "method": "GET",
        "path": "/unknown",
        "body": {}
    }
]

for i, req in enumerate(requests_to_test, start=1):
    print("=" * 60)
    print(f"테스트 요청 {i}")
    print("요청:")
    pprint(req)
    print("응답:")
    pprint(simple_api_server_v2(req))


테스트 요청 1
요청:
{'body': {'message': 'FastAPI가 뭐야?'}, 'method': 'POST', 'path': '/chat'}
응답:
{'data': {'result': "'FastAPI가 뭐야?'에 대한 가짜 LLM 응답입니다."},
 'error': None,
 'status_code': 200,
 'success': True}
테스트 요청 2
요청:
{'body': {}, 'method': 'POST', 'path': '/chat'}
응답:
{'data': None,
 'error': 'message 값이 필요합니다.',
 'status_code': 400,
 'success': False}
테스트 요청 3
요청:
{'body': {}, 'method': 'GET', 'path': '/unknown'}
응답:
{'data': None,
 'error': '요청 경로를 찾을 수 없습니다.',
 'status_code': 404,
 'success': False}


## 실습 정리

이번 실습에서는 서버 내부 처리 흐름을 Python 함수로 구현했습니다.

정리하면 다음과 같습니다.

- 요청 데이터는 dict로 표현할 수 있습니다.
- handler는 입력을 받고 처리 후 응답을 반환합니다.
- 입력 검증은 API 안정성의 기본입니다.
- 응답 구조를 통일하면 클라이언트가 결과를 해석하기 쉬워집니다.
- 오늘 만든 handler 구조는 이후 실제 FastAPI Endpoint 구현으로 확장됩니다.
